# 🌑 LunarSight — Notebook 01: Data Ingestion

**Agent 1**: Downloads DFSAR + LOLA data, reprojects to polar stereographic,
computes terrain slope/aspect, and builds the co-registered tensor.

---

In [ ]:
# === Setup ===
import os

# Mount Google Drive for persistence
from google.colab import drive
drive.mount('/content/drive')

# Clone repo if needed
REPO_DIR = '/content/Lunar-Sight'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/YOUR_USERNAME/Lunar-Sight.git {REPO_DIR}
os.chdir(os.path.join(REPO_DIR, 'Lunar-Sight'))

# Install dependencies
!pip install -q -r requirements_colab.txt

In [ ]:
# === Configuration ===
import yaml
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)s | %(message)s')

with open('config/mission_config.yaml') as f:
    config = yaml.safe_load(f)

print(f"Target: {config['target']['crater_name']}")
print(f"DFSAR URL: {config['data']['dfsar_url']}")

In [ ]:
# === Run Agent 1 ===
from agent1_ingestion.agent import agent1_node

state = {'mission_config_path': 'config/mission_config.yaml'}
result = agent1_node(state)

print(f"Status: {result.get('agent1_status')}")
print(f"Tensor: {result.get('raw_tensor_path')}")
print(f"Slope: {result.get('slope_path')}")

In [ ]:
# === Visualize ===
import numpy as np
import matplotlib.pyplot as plt

if result.get('raw_tensor_path'):
    tensor = np.load(result['raw_tensor_path'])
    fig, axes = plt.subplots(1, min(4, tensor.shape[0]), figsize=(16, 4))
    for i, ax in enumerate(axes):
        im = ax.imshow(tensor[i], cmap='viridis')
        ax.set_title(f'Channel {i}')
        plt.colorbar(im, ax=ax, shrink=0.8)
    plt.suptitle('Co-registered Tensor Channels')
    plt.tight_layout()
    plt.show()

if result.get('slope_path'):
    slope = np.load(result['slope_path'])
    plt.figure(figsize=(8, 8))
    plt.imshow(slope, cmap='RdYlGn_r', vmin=0, vmax=30)
    plt.colorbar(label='Slope (degrees)')
    plt.title('Terrain Slope Map')
    plt.show()